In [ ]:
import sys

print(sys.executable)
print(sys.version)

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import random
import torch

from collections import Counter
from io import BytesIO
from IPython.display import Image, display
from pathlib import Path
from PIL import Image as PILImage, ImageDraw, ImageFont, ImageColor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
SEED = 42
NUM_EPOCHS = 10
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
def generate_structured_JSON(prompts_list, model, device=DEVICE):
  tokenizer = AutoTokenizer.from_pretrained(model)
  model = AutoModelForCausalLM.from_pretrained(
      model,
      device_map="auto",
      torch_dtype=torch.bfloat16
  )

  gemma_outputs = []
  for prompt in prompts_list:
    input_ids = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(**input_ids, max_new_tokens=32)
    gemma_outputs.append(tokenizer.decode(outputs[0]))


  return gemma_outputs

In [ ]:
def create_prompt(examples, ocr_text):
    prompt = f"""You are a Large Language Model acting as an Optical Character Recognition Interpreter.
You will be given raw OCR text from a restaurant receipt:

{ocr_text}

Your goal is to analyze the text and return structured JSON with the following fields:

{{
    "store_name": "<name of the store>",
    "date": "<date of purchase>",
    "subtotal": "<subtotal of the purchase>",
    "tax": "<tax amount>",
    "total": "<total amount>",
    "discount": "<discount amount>",
    "tip": "<tip amount>",
    "service_charge": "<service charge amount>",
    "items": [
        {{
            "item_name": "<name of the item>",
            "quantity": "<quantity of item>",
            "unit_price": "<cost of one item>",
            "total_price": "<total price>",
            "category": "<type of establishment where the item is purchased>"
        }},
        {{
            "item_name": "<name of the item>",
            "quantity": "<quantity of item>",
            "unit_price": "<cost of one item>",
            "total_price": "<total price>",
            "category": "<type of establishment where the item is purchased>"
        }},
        .....,
        {{
            "item_name": "<name of the item>",
            "quantity": "<quantity of item>",
            "unit_price": "<cost of one item>",
            "total_price": "<total price>",
            "category": "<type of establishment where the item is purchased>"
        }}
    ]
}}

Return only a single valid JSON object. It must parse with json.loads as-is.
Do not wrap the JSON in code fences or add commentary.

Example 1:
Raw OCR Text:

{examples[0]["text"]}

Receipt Ground Truth:
{examples[0]["ground_truth"]}

Example 2:
Raw OCR Text:

{examples[1]["text"]}

Receipt Ground Truth:
{examples[1]["ground_truth"]}

Example 3:
Raw OCR Text:

{examples[2]["text"]}

Receipt Ground Truth:
{examples[2]["ground_truth"]}

"""
    return prompt

In [ ]:
fine_Tuning_list = []

In [ ]:
text = """Walmart :'c
Save money. Live better.
(712 ) 366 - 3326
MANAGER DONALD MONEYMAKER
3201 MANAWA CENTRE DR
COUNCIL BLUFFS IA 61601
ST# 01965 008 002031 JE8,20 TRe 061465
SW HRO FGHTR 06305094073: 6.94 T
SUBTOTAL 6.94
TAK 1 7.000 % 0:49
143
‘DEBIT TA 1.43
CHANGE DUE 0.00
EFT DEBIT PAY FROM PRIMARY
7.43 TOTAL PURCHASE
ACCOUNT 4 ok HH soo ES
REF # 534200381765
NETWORK ID. 0056 APPR CODE 087193
TERMINAL # NX996026
12/08/15 12:39:24
# ITEMS SOLD 1
TCR 3261 4114 0323 3051 6498
wii Oia
to midnight 7 daus a week.
as jenis 12:39:26
Savings Catcher! Scan with Walmart app
Ea"""

ground_truth = json.loads("""
{
  "store_name": "Walmart",
  "date": "2015-12-08",
  "subtotal": 6.94,
  "tax": 0.49,
  "total": 7.43,
  "discount": 0,
  "tip": 0,
  "service_charge": 0,
  "items": [
    {
      "item_name": "SW HRO FGHTR",
      "quantity": 1,
      "unit_price": 6.94,
      "total_price": 6.94,
      "category": "Toys and Games"
    }
  ]
}
""")

example = {
    "text": text,
    "ground_truth": ground_truth
}

fine_Tuning_list.append(example)

In [ ]:
text = """
See back of receip'
to win $1000
ID #: 7L219WJM25F
Walmart >/<.
Save money. Live better.
( 701 ) 223 3066
MANAGER TIM MORRIS
2717 ROCK ISLAND PL
BISMARCK ND 58504
ST# 01534 00% 009048 TE# 48 TRH 04455
6 WING PLATE 020108870398 3.98 P
ASST 27 063099656595 4.88 X
CUTIE CAR 063099656644 12.88 X
SUBTOTAL 21.74
TAX 1 6.500 % 1.15
TAX 4 7.500 % 0.30
TOTAL 23.19
DEBIT TEND 23.19
CHANGE DUE 0.00
EFT DEBIT PAY FROM PRIMARY
23.19 TOTAL PURCHASE
US DEBIT Xxxx xxx xxxx 2696 I 0
REF # 731700494718
NETWORK ID. 0081 APPR CODE 936886
US DEBIT
AID A0000000980840
TC 58B77239DA5B3B44
xNO SIGNATURE REQUIRED
TERMINAL # SCO11090
11/13/17 12:34:04
# ITEMS SOLD 3
10% 2188 7274 6444 1567 3762
NUM AT
ii Low Prices You Can Trust. Every Day.
| 11/13/17 12:34:11
Store receipts on your phone. Walmart P
= |
"""

ground_truth = json.loads("""
{
  "store_name": "Walmart",
  "date": "2017-11-13",
  "subtotal": 21.74,
  "tax": 1.45,
  "total": 23.19,
  "discount": 0,
  "tip": 0,
  "service_charge": 0,
  "items": [
    {
      "item_name": "6 WING PLATE",
      "quantity": 1,
      "unit_price": 3.98,
      "total_price": 3.98,
      "category": "Restaurant"
    },
    {
      "item_name": "ASST 27",
      "quantity": 1,
      "unit_price": 4.88,
      "total_price": 4.88,
      "category": "Other"
    },
    {
      "item_name": "CUTIE CAR",
      "quantity": 1,
      "unit_price": 12.88,
      "total_price": 12.88,
      "category": "Toys and Games"
    }
  ]
}
""")

example = {
    "text": text,
    "ground_truth": ground_truth
}

fine_Tuning_list.append(example)

In [ ]:
text = """
WAL*MART
Save money. Live better.
WE SELL FOR LESS
ALWAYS
MANAGER DAPHNE SESSION
ne 636 ) 536 - (aa
IESTERFIELD, MO 6
By 2600" OF 00001855 Te 095 TRE Cr
W FIGURES 065356920144 6.63
TOOTHBRUSH Mes Bira 3 x
WOMEN SLIPPE Ce ATA ope x
Ig . 4)
TAX 1 3 ടാ qe 0.54
TAX 3 4,350.% 076
TOTAL 18.75
MCARD TEND 18.15
ACCOUNT #3546
APPROVAL #007190
TRAN
RANS ID -
VALIDATION -
PAYMENT SERVICE - A
CHANGE DUE 0.00
# ITEMS SOLD 3
AOA
Wal-Mart exclusivi :
available ye stn
10/20/07 13:48:43
##HCUSTOMER COPY##
"""


ground_truth = json.loads("""
{
  "store_name": "Walmart",
  "date": "2007-10-20",
  "subtotal": 17.45,
  "tax": 1.30,
  "total": 18.75,
  "discount": 0,
  "tip": 0,
  "service_charge": 0,
  "items": [
    {
      "item_name": "SW FIGURES",
      "quantity": 1,
      "unit_price": 6.63,
      "total_price": 6.63,
      "category": "Toys and Games"
    },
    {
      "item_name": "TOOTHBRUSH",
      "quantity": 1,
      "unit_price": 0.96,
      "total_price": 0.96,
      "category": "Health and Personal Care"
    },
    {
      "item_name": "WOMEN SLIPPE",
      "quantity": 1,
      "unit_price": 9.86,
      "total_price": 9.86,
      "category": "Clothing"
    }
  ]
}
""")

example = {
    "text": text,
    "ground_truth": ground_truth
}

fine_Tuning_list.append(example)

In [ ]:
create_prompt(fine_Tuning_list, "")

In [ ]:
receipt_dataset_df = pd.read_csv("Receipt_Dataset.csv", usecols=['ground_truth_json' , 'raw_ocr_text' , 'ocr_meta_data'])

ocr_text = receipt_dataset_df['raw_ocr_text'].tolist()
ground_truths = []
for ground_truth in receipt_dataset_df['ground_truth_json'].tolist():
  ground_truths.append(json.loads(ground_truth))


In [ ]:
prompts_list = []

for text in ocr_text:
  tesseract_prompt = create_prompt(fine_Tuning_list, text)
  prompts_list.append(tesseract_prompt)

In [ ]:
from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv()

hf_token = os.getenv("HF_ACCESS_TOKEN")
login(token=hf_token)

#LLama 3.1 8B

In [ ]:
model_id = "meta-llama/Llama-3.1-8B"
llama_outputs = []
for prompt in prompts_list:
  llama_outputs.append(generate_structured_JSON(prompt, model_id))

#Gemma 2 9B

In [ ]:
model_id = "google/gemma-2-9b"
gemma_outputs = generate_structured_JSON(prompts_list, model_id)

In [ ]:
import csv

output_file_path = 'gemma_outputs.csv'
with open(output_file_path, 'w', newline='') as csvfile:
    csv_writer = csv.writer(csvfile)
    csv_writer.writerows(gemma_outputs)

print(f"Output saved to {output_file_path}")

###Evaluation

In [ ]:
import json
from rapidfuzz.distance import Levenshtein

def evaluate_json(predicted, ground_truth):

    results = {}

    for key, actual in ground_truth.items():

        predicted_value = predicted.get(key)

        if predicted_value is None:
            results[key] = {
                "exact_match": 0,
                "similarity": 0
            }
            continue

        # Numeric comparison
        if isinstance(actual, (int, float)) and not isinstance(actual, bool):

            error = abs(float(actual) - float(predicted_value))

            results[key] = {
                "exact_match": int(error < 0.01),
                "absolute_error": error
            }

        # String comparison
        else:

            actual_str = str(actual).lower().strip()
            predicted_str = str(predicted_value).lower().strip()

            similarity = Levenshtein.normalized_similarity(
                actual_str,
                predicted_str
            )

            results[key] = {
                "exact_match": int(actual_str == predicted_str),
                "similarity": similarity
            }

    return results

In [ ]:
results = []
for llm_prediction, ground_truth in zip(gemma_outputs, ground_truths):
  result = evaluate_json(llm_prediction, llm_prediction)
  #print(json.dumps(results, indent=4))
  results.append(result)

print("Percentage Exact Match: " )

In [ ]:
count_contains_same_keys = 0
overall_precision = 0
for llm_prediction, ground_truth in zip(gemma_outputs, ground_truths):
  if isinstance(llm_prediction, dict) and isinstance(ground_truth, dict):
    if llm_prediction.keys() == ground_truth.keys():
      count_contains_same_keys = count_contains_same_keys + 1
      correct_values = 0
      for key in llm_prediction.keys():
        if llm_prediction[key] == ground_truth[key]:
          correct_values = correct_values + 1
      overall_precision = overall_precision + (correct_values / len(llm_prediction.keys()))

overall_precision = overall_precision / len(gemma_outputs)
